In [ ]:
## fonctions de base

#packages

using Pkg
#Pkg.add("LinearAlgebra")
#Pkg.add("LinearSolve")
#Pkg.add("Plots")
#Pkg.add("Printf")
#Pkg.add("LaTeXStrings")
#Pkg.add("Interpolations")
#Pkg.add("NPZ")


using LinearAlgebra
using LinearSolve
using Plots
using LaTeXStrings
using Interpolations
using NPZ
using Printf


#-------------------------------------------------------------------------------------
#	dressing of function f (given as array f_discr, occupation ratio given as n_discr)
#-------------------------------------------------------------------------------------


function f_varphimat(lam_discr,gb)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)
    return varphimat
end 

function f_dlam(lam_discr)
    L = size(lam_discr)[1]
    dlam = zeros(L)
    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    return dlam
end
    
function dress(gb, lam_discr, n_discr, f_discr)
    
    
    varphimat = f_varphimat(lam_discr,gb)
    dlam = f_dlam(lam_discr)
    
    A = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    A\f_discr
end

#-------------------------------------------------------------------------------------
#	to evaluate charge density associated with function f
#-------------------------------------------------------------------------------------

function charge_density(gb, lam_discr, n_discr, f_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    A = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    0.5/pi * dot(dlam , Diagonal(n_discr) * (A \ f_discr) )
end

#-------------------------------------------------------------------------
#	solve Yang-Yang equation
#-------------------------------------------------------------------------

function fun1(z)
    if z>0
        return log(1. +exp(-z))
    else
        return log(1. +exp(z)) - z
    end
end

function fun2(z)
    if z<0
        return 1. /(1. +exp(z))
    else
        return exp(-z)/(1. +exp(-z))
    end
end

function yangyang(gb, beta, lam_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])

    bare_E = 0.5 * beta[3] * lam_discr.^2 + beta[2] * lam_discr + beta[1] * ones(L)
    eps = copy(bare_E)
    n = fun2.(eps)

    #iterative solution
    diff = 1.
    while diff>1e-12
        old_n = copy(n)
        eps = bare_E - 0.5/pi * varphimat*Diagonal( fun1.(eps) )*dlam
        n = fun2.(eps)
        diff = norm(n-old_n)
    end
    eps = bare_E - 0.5/pi * varphimat*Diagonal( fun1.(eps) )*dlam
    n = fun2.(eps)
    return n
end

#-------------------------------------------------------------------------------------
#	compute effective velocity
#-------------------------------------------------------------------------------------

function veff(gb, lam_discr, n_discr)
    L = size(lam_discr)[1]
    varphi(lam) = 2*gb/( gb^2 + lam^2 )
    varphimat = zeros(L,L)
    dlam = zeros(L)

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( lam_discr[i]-lam_discr[j] )
            varphimat[j,i] = varphi( lam_discr[i]-lam_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)

    for i = 2:L-1
        dlam[i] = 0.5*(lam_discr[i+1]-lam_discr[i-1])
    end
    dlam[1] = 0.5*(lam_discr[2]-lam_discr[1])
    dlam[L] = 0.5*(lam_discr[L]-lam_discr[L-1])
    dresser = I - 0.5/pi * varphimat*Diagonal(n_discr)*Diagonal(dlam)
    onedr = dresser\ones(L)
    iddr = dresser\copy(lam_discr)
    iddr ./ onedr
end

#-------------------------------------------------------------------------------------
#	time evolution for box expansion
#-------------------------------------------------------------------------------------

function evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
    npts_integral = 600
    L = size(x1)[1]
    
    veff1 = zeros(L)
    veff2 = zeros(L)
    #println(@sprintf "size x2 = %.3f , size theta_tab  = %.3f " size(x1)[1] size(theta_tab)[1])
    #display(x2)
    #display(theta_tab)
    f2 = linear_interpolation(x2, theta_tab)
    f1 = linear_interpolation(x1, theta_tab)
    
    f2 = interpolate((x2,), theta_tab, Gridded(Linear()))
    f2 = extrapolate(f2, Linear()) 
    
    f1 = interpolate((x1,), theta_tab, Gridded(Linear()))
    f1 = extrapolate(f1, Linear())
    #velocity of first and last points is the one of free particles
    veff1[1] = theta_tab[1]
    veff2[L] = theta_tab[L]

    for j in 2:L
        thet1 = theta_tab[j]
        if x1[j]<x2[1]
            thet2 = theta_tab[1]
        else
            thet2 = f2(x1[j])
        end
        lam_discr = LinRange(thet2,thet1,npts_integral)
        n_discr = n_fun.(lam_discr)
        veff1[j] = last(veff(gb, lam_discr, n_discr))
        #velocities of right contour (using parity symmetry)
        #veff2[L+1-j] = -veff1[j]
    end
    # sans symetrie
    for j in 1:L-1
        thet1 = theta_tab[j]
        if x2[j] > x1[L]
            thet2 = theta_tab[L]
        else 
            thet2 = f1(x2[j])
        end 
        lam_discr = LinRange(thet1, thet2, 600)
        n_discr = n_fun.(lam_discr)
        veff2[j] = veff(gb, lam_discr, n_discr)[1]
    end
    
    return x1 + dt*veff1, x2 + dt*veff2
end

#-------------------------------------------------------------------------------------
#	box expansion: calculate density (careful: return density in units of rapidities
#    for density in mu^-1 multiply by mass/hbar)
#-------------------------------------------------------------------------------------

function eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
    L = size(x1)[1]
    
    dens = zeros(2,2*L)
    f2 = linear_interpolation(x2, theta_tab)
    f2 = interpolate((x2,), theta_tab , Gridded(Linear()))
    f2 = extrapolate(f2, Linear())
    f1 = linear_interpolation(x1, theta_tab)
    f1 = interpolate((x1,), theta_tab , Gridded(Linear()))
    f1 = extrapolate(f1, Linear())
    #density of first and last points is zero
    dens[1,1] = x1[1]
    dens[2,1] = 0
    dens[1,L+1] = x2[L]
    dens[2,L+1] = 0
    #println( @sprintf "n_p = %.3f" ratio_m_hbar * charge_density(gb, theta_tab, n_fun.(theta_tab), ones(size(theta_tab)[1]) ))
    for j in 2:L
        thet1 = theta_tab[j]
        if x1[j]<x2[1]
            thet2 = theta_tab[1]
        else
            #println("nnn")
            thet2 = f2(x1[j])
        end
        lam_discr = LinRange(thet2,thet1,600)
        n_discr = n_fun.(lam_discr)
        dens[1,j] = x1[j]
        dens[2,j] = ratio_m_hbar * charge_density(gb, lam_discr, n_discr, ones(size(lam_discr)[1]) )
        #if j == L 
        #    println( @sprintf " j = %.3f , theta 1 = %.3f , theta 2 = %.3f , n_p1 = %.3f" j thet2 thet1 dens[2,j])
        #end 
        #velocities of right contour (using parity symmetry)
        #dens[1,L+j] = x2[L+1-j]      
        #dens[2,L+j] = dens[2,j]
    end
    #without symmetry 
    for j in 1:L-1
        thet1 = theta_tab[j]
        if x2[j]>x1[L]
            thet2 = theta_tab[L]
        else 
            #println("nnn")
            thet2 = f1(x2[j])
        end 
        lam_discr = LinRange(thet1,thet2,600)
        n_discr = n_fun.(lam_discr)
        dens[1,2*L+1-j] = x2[j] 
        dens[2,2*L+1-j] = ratio_m_hbar * charge_density(gb, lam_discr, n_discr, ones(size(lam_discr)[1]) ) 
        #if j == 1 
        #    println( @sprintf " j = %.3f , theta 1 = %.3f , theta 2 = %.3f , n_p1 = %.3f" j thet1 thet2 dens[2,2*L+1-j])
        #end 
    end
    #sort densities according to positions, and return
    dens[:,sortperm(dens[1, :])]
end

#------------------------------------------------------------------------------------------------
#	function for simulating box expansion. Writes results in file 'data/density_expansion_t.npd'
#     where 't' is the time (takes a list of times to be saved as argument)
#------------------------------------------------------------------------------------------------

function box_expansion(n_fun, dt, list_t_save, theta_max, npts, gb, ratio_m_hbar)
    tmax = maximum(list_t_save)
    theta_tab = LinRange(-theta_max, theta_max, npts)
    x1 = -0.5 * ones(npts) + 0.0000001*theta_tab
    x2 = 0.5 * ones(npts) + 0.0000001*theta_tab
        
    t = 0
    while t<=tmax
        x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
        t += dt
        
        for ts in list_t_save
            if t-dt<ts && ts<=t
                mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                println(@sprintf "t = %.3f saved" t)
                #filename = @sprintf "%s/density_expansion_n_theta_etoile_conv_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mu) Float64(T) Float64(Taille) Float64(t)
                #filename = @sprintf "%s/density_expansion_n_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mu) Float64(T) Float64(Taille) Float64(t)
                filename = @sprintf "%s/density_expansion_n_theta_etoile_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t)
                #filename = @sprintf "%s/density_expansion_n_tout_py_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t)
                npzwrite(filename, mydensity)
            end
        end
    end
end



In [ ]:
function DeltaMat(theta_discr , gbar )
    L = size(theta_discr)[1]
    varphi(theta) = 2*gbar/( gbar^2 + theta^2 )
    varphimat = zeros(L,L)
    

    for i = 1:L
        for j = 1:i
            varphimat[i,j] = varphi( theta_discr[i]-theta_discr[j] )
            varphimat[j,i] = varphi( theta_discr[i]-theta_discr[j] )
        end
    end
    varphimat = Symmetric(varphimat)
    return varphimat
end 

function Delta(theta_discr)
    L = size(theta_discr)[1]
    dtheta = zeros(L)
        
    for i = 2:L-1
        dtheta[i] = 0.5*(theta_discr[i+1]-theta_discr[i-1])
    end
    dtheta[1] = 0.5*(theta_discr[2]-theta_discr[1])
    dtheta[L] = 0.5*(theta_discr[L]-theta_discr[L-1])
    
    return dtheta
end 

function montrer_les_rho(theta_discr, nu_discr)
    L = length(theta_discr)
    
    varphimat = DeltaMat(theta_discr , gbar)
    dtheta = Delta(theta_discr)
    
    # Plot nu_tilde
    p = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="f", legend=:topright)
    plot!(p , theta_discr , nu_discr, label="\$\\tilde{\\nu}\$")
    # Méthode 0
    rho_s_0 = 0.5 / pi * dress(gbar , theta_discr, nu_discr, ones(L))
    rho_0 = nu_discr .* rho_s_0
    
    # Méthode 1
    Mat_F_1 = I - (1 / (2 * pi)) * Diagonal(dtheta) * Diagonal(nu_discr) * varphimat
    B = (1 / (2 * pi)) * nu_discr
    rho_1 = Mat_F_1 \ B
    rho_s_1 = rho_1 ./ nu_discr

    # Méthode 2
    Mat_F_2 = I - 0.5 / pi * varphimat * Diagonal(nu_discr) * Diagonal(dtheta)
    B = (1 / (2 * pi)) * ones(L)
    rho_s_2 = Mat_F_2 \ B
    rho_2 = nu_discr .* rho_s_2
    
    # Plot résultats de la méthode 0
    plot!(p, theta_discr, rho_s_0, label="Meth 0 : \$\\tilde{\\rho_s}= \\frac{1}{2\\pi}\\\$")
    plot!(p, theta_discr, rho_0, label="Meth 0 : \$\\tilde{\\rho} = \\tilde{\\nu} * \\tilde{\\rho_s}\$")
    
    # Plot résultats de la méthode 1
    plot!(p, theta_discr, rho_1, label="Meth 1 : \$\\tilde{\\rho}= \\frac{\\tilde{\\nu}}{2\\pi} ( 1 + \\tilde{\\Delta} * \\tilde{\\rho}) \$")
    plot!(p, theta_discr, rho_s_1, label="Meth 1 : \$\\tilde{\\rho}_s  = \\frac{\\tilde{\\rho}}{\\tilde{\\nu}}  \$")
    
    # Plot résultats de la méthode 2
    plot!(p, theta_discr, rho_s_2, label="Meth 2 : \$\\tilde{\\rho_s}= \\frac{1}{2\\pi}\$")
    plot!(p, theta_discr, rho_2, label="Meth 2 : \$\\tilde{\\rho} = \\tilde{\\nu} * \\tilde{\\rho_s}\$")

    display(p)
    
    #p1 = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="\$\\Delta\$", legend=:topright)

    # Plot des différences entre les résultats des deux méthodes
    #plot(p1,theta_discr, rho_s_1 - rho_s_2, label="\$ \\Delta \\tilde{\\rho_s}\$")
    #plot(p1,theta_discr, rho_1 - rho_2, label="\$\\Delta \\tilde{\\rho}\$")

    # Légendes et labels
    #display(p1)
end

#-------------------------------------------------------------------------------------
#	theta start
#-------------------------------------------------------------------------------------
#Pkg.add("Roots")
using Roots

function find_nu_theta_star_discr( theta_discr , theta_star  , nu_discr )
    return ifelse.(theta_discr .> theta_star, 0, nu_discr)
end 

function find_theta_star(theta_discr, nu_discr ,  gbar , v)
    
    # Définition de la fonction f_veff_star_theta_star
    f_veff_star_theta_star = theta_star -> linear_interpolation(theta_discr, veff( gbar, theta_discr, find_nu_theta_star_discr(theta_discr, theta_star, nu_discr)))(theta_star)-v
    
    return fzero(f_veff_star_theta_star, theta_discr[1] , last(theta_discr))
end

In [ ]:
import Pkg
#Pkg.add("Dates")
using Dates

# Formater la date selon le format spécifié
new_date = Dates.format(Dates.today(), "yyyy-mm-dd")

# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)

In [ ]:
new_date = "analyse_2024-11-11"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)

In [ ]:
using Plots
using Printf
using Interpolations

# Définir les constantes
hbar = 1.05457182e-25   # um^2.kg/ms
mass =  1.44e-25        # kg (mass of Rubidium 87)
kB = 1.380649e-26       # um^2.ms^-2.kg.nK^{-1}
a3D = 5.3e-3            # um
om_perp = 2 * pi * 2.56  # kHz (transverse frequency)

# Calculer les constantes dérivées
g = 2 * hbar * a3D * om_perp        # effective 1d repulsion strength
c = mass / hbar^2 * g              # um^{-1}
gbar = g / hbar                   # um.ms^{-1}

In [ ]:
theta_max = 20
npts = 200
theta_discr = LinRange(-theta_max, theta_max , npts) 


#------------------------------------------------------------------------------------------------
#	Fonction pour calculer les nu , rho et rho_s et predire cest fonction apres la coupure 1 
#-
function save_nu_rho_rho_s(new_date , nom , theta_discr, n_p  , mup , T , x0 , Taille , t_deform,  gbar )
    if mup == "None" 
       mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
    else 
        n_p = (((mup / (hbar * om_perp)) + 1)^2 - 1) / (4 * a3D)
    end
    # Discrétisation de theta
    beta = [-mup / (kB * T) 0 mass * g^2/(kB * T * hbar^2)]
    beta = [-mup / (kB * T) 0 mass /(kB * T)]

    # Définir la date sous forme de chaîne

    index_last_underscore = findlast(isequal('_'), nom)

    
    filename = @sprintf "%s/theta_discr_%s.npz" new_date nom[index_last_underscore+1:end]
    npzwrite(filename, theta_discr )
    println("saved : ", filename)  

    @time nu_discr = yangyang(gbar, beta, theta_discr)

    filename = @sprintf "%s/nu_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, nu_discr )
    println("saved : ", filename)
    #montrer_les_rho(theta_discr , nu_discr)

    @time rho_s_discr = 0.5 / pi * mass/hbar*dress(gbar, theta_discr, nu_discr, ones(size(theta_discr)[1]))
    filename = @sprintf "%s/rho_s_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_s_discr )
    println("saved : ", filename)

    @time rho_discr = nu_discr .* rho_s_discr
    filename = @sprintf "%s/rho_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, rho_discr )
    println("saved : ", filename)
    
    ## veff de bord j'ai 3 metdode qui donnent la meme foction à voir 
    @time veff_discr = veff( gbar, theta_discr, nu_discr)
    filename = @sprintf "%s/veff_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, veff_discr )
    println("saved : ", filename)
    
        
    #find_nu_theta_star_discr( theta_discr , theta_star  , nu_discr )return ifelse.(theta_discr .> theta_star, 0, nu_discr)
    function f_veff_star_theta_star(theta_star)
        theta_max = last(theta_discr)
        #println(@sprintf "max : %.3f"  maximum(theta_discr))
        #println("mim : " , minimum(theta_discr))    
        if theta_star == -theta_max 
            return -theta_max
        end 
        n_discr_3 = find_nu_theta_star_discr( theta_discr , theta_star  , nu_discr )
        #plot!(p, theta_discr,  n_discr_3, label="n_{$(theta_star)}")
        #plot!(p1, theta_discr,  veff( gbar, theta_discr, n_discr_3), label="v_{$(theta_star)}")
        veff_fun = LinearInterpolation(theta_discr,veff( gbar, theta_discr, n_discr_3)  )
        return veff_fun(theta_star)
    end 
        
    
    #println( @sprintf "veff (%.3f) = %.3f" theta_star f_veff_star_theta_star_2(theta_star))
    #veff_discr_2 = [f_veff_star_theta_star_2(theta) for theta in theta_discr]
        
    L = size(theta_discr)[1]    
    veff_discr_4 = zeros(L)
    #p = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="n", legend=:topright)
    #p1 = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="veff", legend=:topright)
    for i = 1 : L 
        veff_discr_4[i] = f_veff_star_theta_star(theta_discr[i])
    end 
    #display(p)
    #display(p1)

    
    #@time veff_discr_3 = [theta_star == theta_max ? theta_max : veff(gbar, LinRange(theta_star, theta_max, 600), LinearInterpolation(theta_discr, nu_discr).(LinRange(theta_star, theta_max, 600)))[1] for theta_star in theta_discr]
    filename = @sprintf "%s/veff_bord_discr_%s_%.3f_%.3f.npz" new_date nom[index_last_underscore+1:end] mup/kB T
    npzwrite(filename, veff_discr_4 )
    #println("saved : ", filename)
        
    #p = plot(xlabel="\$\\tilde{\\theta}\$", ylabel="veff", legend=:topright)
    #plot!(p, theta_discr, veff_discr_4, label="v bord 4")
    #display(p)
    
    v = x0 /t_deform       
            
    f = theta_star -> f_veff_star_theta_star(theta_star)-v
    
    @time theta_star = fzero(f, theta_discr[1] , last(theta_discr))
    filename = @sprintf "%s/%s_%.3f_%.3f_%.3f.npz" new_date nom mup/kB T x0
    npzwrite(filename, theta_star )
    println("saved : ", filename)

    @time nu_theta_star_discr = find_nu_theta_star_discr(theta_discr , theta_star  , nu_discr )
    filename = @sprintf "%s/nu_%s_discr_%s_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0
    nu_theta_star_discr = Float64.(nu_theta_star_discr)
    npzwrite(filename, nu_theta_star_discr )
    println("saved : ", filename)
    

    println(@sprintf "theta_star = %.3f " theta_star)



    @time rho_s_theta_star_discr = 0.5 / pi * mass/hbar * dress(gbar, theta_discr, nu_theta_star_discr, ones(size(theta_discr)[1]))
    filename = @sprintf "%s/rho_s_%s_discr_%s_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0
    npzwrite(filename, rho_s_theta_star_discr )
    println("saved : ", filename)

    @time rho_theta_star_discr = nu_theta_star_discr .* rho_s_theta_star_discr
    filename = @sprintf "%s/rho_%s_discr_%s_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0
    npzwrite(filename, rho_theta_star_discr )
    println("saved : ", filename)
    
    #println("moyenn")
    Deltav = Taille/t_deform
    nbsv = 20
    V  = LinRange(v-Deltav/2 , v+Deltav/2 , nbsv )#npts)
    nbsv = size(V)[1]
    dv = Deltav/nbsv
    
    nu_theta_discr_moy = zeros(L)
    rho_s_theta_discr_moy = zeros(L)
    rho_theta_discr_moy = zeros(L)
    
    nbsv = size(V)[1]
    for i = 1 : nbsv
        f = theta_star -> f_veff_star_theta_star(theta_star)-V[i]
        theta_star = fzero(f, theta_discr[1] , last(theta_discr))
        #println(@sprintf "theta_star = %.3f " theta_star)
        nu_theta_star_discr_e = find_nu_theta_star_discr(theta_discr , theta_star  , nu_discr )
        nu_theta_discr_moy += nu_theta_star_discr_e
        rho_s_theta_star_discr_e = 0.5 / pi * mass/hbar * dress(gbar, theta_discr, nu_theta_star_discr_e, ones(size(theta_discr)[1]))
        rho_s_theta_discr_moy += rho_s_theta_star_discr_e
        rho_theta_discr_moy += nu_theta_star_discr_e .* rho_s_theta_star_discr_e
    end 
    
    nu_theta_discr_moy = nu_theta_discr_moy*dv/Deltav
    rho_s_theta_discr_moy = rho_s_theta_discr_moy*dv/Deltav
    rho_theta_discr_moy = rho_theta_discr_moy*dv/Deltav
    
    filename = @sprintf "%s/nu_%s_discr_moy_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0 Taille
    nu_theta_discr_moy = Float64.(nu_theta_discr_moy)
    npzwrite(filename, nu_theta_discr_moy )
    #println("saved : ", filename)
    
    filename = @sprintf "%s/rho_s_%s_discr_moy_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0 Taille
    npzwrite(filename, rho_s_theta_discr_moy )
    #println("saved : ", filename)

    filename = @sprintf "%s/rho_%s_discr_moy_%s_%.3f_%.3f_%.3f_%.3f.npz" new_date nom[1:index_last_underscore-1] nom[index_last_underscore+1:end] mup/kB T x0 Taille
    npzwrite(filename, rho_theta_discr_moy )
    #println("saved : ", filename)
        
        
    
    
    return theta_discr , nu_discr , rho_s_discr , rho_discr , veff_discr , veff_discr_4 ,  
                            theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr , nu_theta_discr_moy, rho_s_theta_discr_moy, rho_theta_discr_moy 

end 

new_date = "test_1"
if !isdir(new_date)
    mkdir(new_date)
    end
nom = "test_1"

    
# Afficher la date formatée
println(new_date)
mup = 50 * kB
n_p = "None"
println(n_p)
T = 100                           
theta_discr , nu_discr , rho_s_discr , rho_discr ,veff_bord, veff_bord_coup ,
    theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr, nu_theta_star_discr_e, rho_s_theta_star_discr_e, rho_theta_discr_moy = 
    save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T , 20, 30 ,18 ,  gbar )


In [ ]:
using DelimitedFiles




function f_DONNEES(date_donnees)
    DONNEES = []
    
    if date_donnees == "None"
        fichier_x = "Scan43_coupure2_1ms_x.txt"
        fichier_y = "Scan43_coupure2_1ms_n.txt"
        Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "2em coupure 1ms"]
        push!(DONNEES, Donnees)
        fichier_x = "Scan43_coupure1_10ms_x.txt"
        fichier_y = "Scan43_coupure1_10ms_n.txt"
        Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "1er coupure 10 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "Scan43_x.txt"
        fichier_y = "Scan43_n.txt"
        Donnees = [readdlm("2024-02-05/Donnees/$(fichier_x)") , reverse(readdlm("2024-02-05/Donnees/$(fichier_y)")) * 0 , "2em coupure 40 ms"]
        push!(DONNEES, Donnees)
    elseif date_donnees == "2024-02-05"
        fichier_x = "Scan43_coupure2_1ms_x.txt"
        fichier_y = "Scan43_coupure2_1ms_n.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "2em coupure 1ms"]
        push!(DONNEES, Donnees)
        fichier_x = "Scan43_coupure1_10ms_x.txt"
        fichier_y = "Scan43_coupure1_10ms_n.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "1er coupure 10 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "Scan43_x.txt"
        fichier_y = "Scan43_n.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , reverse(readdlm("$(date_donnees)/Donnees/$(fichier_y)")) , "2em coupure 40 ms"]
        push!(DONNEES, Donnees)
    elseif date_donnees == "2024-02-09"
        fichier_x = "X_coupure1_1ms.txt"
        fichier_y = "Y_coupure1_1ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure1_18ms.txt"
        fichier_y = "Y_coupure1_18ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 18 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_1ms.txt"
        fichier_y = "Y_coupure2_1ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_50ms.txt"
        fichier_y = "Y_coupure2_50ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 50 ms"]
        push!(DONNEES, Donnees)
    elseif date_donnees == "2024-02-29" 
        fichier_x = "X_coupure1_1ms.txt"
        fichier_y = "Y_coupure1_1ms.txt"
        Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure1_28ms.txt"
        fichier_y = "Y_coupure1_28ms.txt"
        Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 28 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_1ms.txt"
        fichier_y = "Y_coupure2_1ms.txt"
        Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_50ms.txt"
        fichier_y = "Y_coupure2_50ms.txt"
        Donnees = [-readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 50 ms"]
        push!(DONNEES, Donnees)
    elseif date_donnees == "2024-04-24"
        fichier_x = "X_coupure1_1ms.txt"
        fichier_y = "Y_coupure1_1ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure1_18ms.txt"
        fichier_y = "Y_coupure1_18ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 1 18 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_1ms.txt"
        fichier_y = "Y_coupure2_1ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 1 ms"]
        push!(DONNEES, Donnees)
        fichier_x = "X_coupure2_30ms.txt"
        fichier_y = "Y_coupure2_30ms.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "coupure 2 30 ms"]
        push!(DONNEES, Donnees)    
        fichier_x = "X_insitut.txt"
        fichier_y = "Y_insitut.txt"
        Donnees = [readdlm("$(date_donnees)/Donnees/$(fichier_x)") , readdlm("$(date_donnees)/Donnees/$(fichier_y)") , "insitut"]
        push!(DONNEES, Donnees)                         
    end
    return DONNEES
end 

date_donnees = "2024-04-24"
DONNEES = f_DONNEES(date_donnees)

for i in 1:length(DONNEES)
    p = plot(xlabel="x", ylabel="y", legend=:topright)                           
    x , y , label = DONNEES[i]    
    plot!(p, x,  y, label = label)
    display(p)                         
end                             

# Bord ( 2em coupure homogène)

In [ ]:
function pre_model_fit_homo(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    
    function box_expansion(n_fun, dt, list_t_save, theta_max, npts, gb, ratio_m_hbar)
        
        filename0 = @sprintf "%s/density_expansion_nu_julia_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(Taille)
        filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(Taille)
    
        tmax = maximum(list_t_save)
        theta_tab = LinRange(-theta_max, theta_max, npts)
        x1 = -0.5 * ones(npts) + 0.0000001*theta_tab
        x2 = 0.5 * ones(npts) + 0.0000001*theta_tab

        t = 0

        mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t))
        println("saved : ", filename)
        npzwrite(filename, mydensity)

        #p = plot(xlabel="x", ylabel="n", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:] )
        #display(p)
        while t<tmax
            x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
            t += dt

            for ts in list_t_save
                if ts <= t < (ts + dt)
                    mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                    println(@sprintf "t = %.3f saved" t*Taille)
                    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                    println("saved : ", filename)
                    npzwrite(filename, mydensity)

                    #p = plot(xlabel="x", ylabel="n", legend=:topright)
                    #plot!(p, mydensity[1,:],  mydensity[2,:] )
                    #display(p)
                end
            end
        end  
        
        p = plot(xlabel="x", ylabel="n", legend=:topright)
        plot!(p, mydensity[1,:],  mydensity[2,:] )
        display(p)
        return mydensity[1,:],  mydensity[2,:]
    end
    
    theta_discr , nu_discr , rho_s_discr , rho_discr , veff_bord , veff_bord_coup ,theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr = save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    # Créer l'interpolation
    #@time nu_fun = LinearInterpolation(theta_discr, nu_discr)
    @time nu_fun = LinearInterpolation(theta_discr, nu_theta_star_discr)

    # Afficher l'interpolation
    #p = plot(xlabel="theta", ylabel="nu", legend=:topright)
    #plot!(p, theta_discr,  nu_discr )
    #plot!(p, theta_discr, nu_fun(theta_discr))
    #display(p)
    
    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille ]
    temps_sav = [ 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille, 50.0/Taille ]
    dt = 0.9/Taille
    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]

    #temps_sav = [ 50.0/Taille  ]
    dt = 1/Taille
    return box_expansion(nu_fun, dt ,temps_sav  , theta_max , npts , gbar, mass/hbar)
end

nom = "theta_star_1.0_h"
theta_discr = LinRange(-theta_max, theta_max , npts) 

function model_fit_homo(x , p )
    mup , T = p[1]*kB , p[2]
    n_p = "None"
    println("mup = ", mup/kB)
    println("T = ", T)  
    
    x_simul , y_simul = pre_model_fit_homo(new_date , nom , theta_discr, n_p , mup ,  T , Taille , gbar )
    density_fun = LinearInterpolation(x_simul , y_simul) 
    p = plot(xlabel="xxx", ylabel="n", legend=:topright)
    plot!(p, x,  last(DONNEES)[2])
    plot!(p, x,  density_fun.(x))
    display(p)
    return Float64.(density_fun.(x))
end 

# Bord (2em coupure ihomogène)

## evolution d'un bord 

In [ ]:
#-------------------------------------------------------------------------------------
#	time evolution for box expansion
#-------------------------------------------------------------------------------------

function evol_edge_expansion(dt, x, gb, theta_tab, n_fun)
    npts_integral = 600
    L = size(x)[1]
    
    veff_tab = zeros(L)

    f = linear_interpolation(x, theta_tab)

    #velocity of first and last points is the one of free particles
    veff_tab[1] = theta_tab[1]

    for j in 2:L
        thet1 = theta_tab[j]
        thet2 = theta_tab[1]
        lam_discr = LinRange(thet2,thet1,npts_integral)
        n_discr = n_fun.(lam_discr)
        #println("vell")
        #@time veff_tab[j] = last(veff(gb, lam_discr, n_discr))
        veff_tab[j] = last(veff(gb, lam_discr, n_discr))
        #velocities of right contour (using parity symmetry)
    end
    
    return x + dt*veff_tab
end

function evol_edge_expansion_balistic(dt, x, gb, theta_tab, n_fun)
    npts_integral = 600
    L = size(x)[1]
    
    veff_tab = zeros(L)

    f = linear_interpolation(x, theta_tab)

    #velocity of first and last points is the one of free particles
    veff_tab[1] = theta_tab[1]

    for j in 2:L
        thet1 = theta_tab[j]
        thet2 = theta_tab[1]
        lam_discr = LinRange(thet2,thet1,npts_integral)
        n_discr = n_fun.(lam_discr)
        veff_tab[j] = last(veff(gb, lam_discr, n_discr))
        #velocities of right contour (using parity symmetry)
    end
    
    return x + dt*veff_tab
end

#-------------------------------------------------------------------------------------
#	box expansion: calculate density (careful: return density in units of rapidities
#    for density in mu^-1 multiply by mass/hbar)
#-------------------------------------------------------------------------------------

function eval_edge_density(x, gb, theta_tab, nu_fun, ratio_m_hbar)
    npts_integral = 600
    
    L = size(x)[1]
    dens = zeros(2,L)
    dens[1,1] = x[1]
    dens[2,1] = 0
    thet1 = theta_tab[1]


    for j in 2:L
        thet1 = theta_tab[j]
        thet2 = theta_tab[1]
        lam_discr = LinRange(thet2,thet1,npts_integral)
        nu_discr = nu_fun.(lam_discr)
        dens[1,j] = x[j]      
        dens[2,j] = ratio_m_hbar * charge_density(gb, lam_discr, nu_discr, ones(size(lam_discr)[1]) )
    end
    
    println( "la densité haute est : " , dens[2,L])

    #sort densities according to positions, and return
    return dens[:,sortperm(dens[1, :])] #, nu
end



In [ ]:
function pre_model_fit_inhomo(new_date , nom,  theta_discr, n_p , mup ,  T ,  x0 , Taille , gbar , mode , x , y  )
    
    calcule_deformation, fit_deformation, calcule_expension, fit_expension = mode
    
    nom = "theta_edge_1.0_ih1"
    t_deforme = 18
    
    if calcule_deformation
        
        if fit_deformation && n_p != "None"
            mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
            println( "mup : " , mup , "    " ,  mup/kB )
        end
        
        
        
        theta_discr , nu_discr , rho_s_discr , rho_discr ,veff_bord, veff_bord_coup ,
        theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr, nu_theta_discr_moy, rho_s_theta_discr_moy, rho_theta_discr_moy = 
        save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T ,x0, Taille , t_deforme, gbar )

        filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)
        filename1 = @sprintf "%s/nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)

        function edge_expansion(n_fun, theta_max, npts, gb, ratio_m_hbar)
            theta_tab = LinRange(-theta_max, theta_max, npts)
            x = 0.0000001*theta_tab
            
            
            t = 0
            mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
            println(@sprintf "t = %.3f saved" t*Taille)
            filename = filename0 * @sprintf("%.3f.npz", Float64(t))
            println("saved : ", filename)
            npzwrite(filename, mydensity)
            filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(t*Taille)
            println("saved : ", filename)
            npzwrite(filename, x)

            p = plot(xlabel="x", ylabel="n", legend=:topright)
            plot!(p, mydensity[1,:],  mydensity[2,:] , label = " t = 0.0000001" )
            #display(p)
            
            t = "veff_bord"
            x += veff_bord_coup
            #filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f_%s.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) t)
            #println("saved : ", filename)
            #npzwrite(filename, x)
            mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
            println(@sprintf "t = %s saved" t)
            filename = filename0 * @sprintf("%s.npz", t)
            println("saved : ", filename)
            npzwrite(filename, mydensity)
            
 
            #p = plot(xlabel="v", ylabel="n", legend=:topright)
            plot!(p, mydensity[1,:],  mydensity[2,:]  , label = " t = hydro ")
            #display(p)
        end

        # Créer l'interpolation
        nu_fun = LinearInterpolation(theta_discr, nu_discr)

        # Afficher l'interpolation
        p1 = plot(xlabel="theta", ylabel="nu", legend=:topright)
        #plot!(p1, theta_discr,  nu_discr , label = "nu_discret" )
        #plot!(p1, theta_discr, nu_fun(theta_discr) , label = "nu_fun(theta_discr)")
        #display(p1)

        edge_expansion(nu_fun, theta_max , npts , gbar, mass/hbar)

        ###Coupure de taile $\ell$ + evolution
        t = "veff_bord"
        filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)
        filename = filename0 * @sprintf("%s.npz", t)
        #filename = @sprintf "%s/density_expansion_nu_edge_julia_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
        density = npzread(filename)

        

        x1 = density[1,:]#*t_deforme#/Taille
        

        plot!(p,  x1 , density[2,:] , label = " t = 18 ")
        display(p)
        
        if fit_deformation 
            return x1 , density[2,:] 
        end
    end 
    
    if calcule_expension 
   

        nom = "theta_edge_1.0_ih2"
        index_last_underscore = findlast(isequal('_'), nom)
        println(nom[index_last_underscore+1:end])
        
        println("saveeeeeeeee")
        #save_nu_rho_rho_s(new_date , nom , theta_discr_2, n_p , mup , T , Taille , gbar )
        theta_discr , nu_discr , rho_s_discr , rho_discr ,veff_bord, veff_bord_coup ,
        theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr, nu_theta_discr_moy, rho_s_theta_discr_moy, rho_theta_discr_moy = 
        save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T ,x0, Taille , t_deforme, gbar )


        
        x_fun = LinearInterpolation(theta_discr,x1)
        theta_fun = LinearInterpolation(x1, theta_discr)
        x_1 , x_2= (x0 - Taille/2)/t_deforme , (x0 + Taille/2)/t_deforme
        #x_1 , x_2 = 0.43393268627490844, 1.7461461601247348#0.38599379698434566 , 1.7092275073793108#7.2307096175967445/t_deforme  , 32.01070961759683/t_deforme
        
        theta_1 , theta_2 = theta_fun(x_1), theta_fun(x_2)
        max_abs_theta = maximum([20 , abs(theta_1) , abs(theta_2)])
        
        theta_bord1 = LinRange(-max_abs_theta, theta_1 , 100)
        x1_bord1 = x_1*ones(size(theta_bord1)[1])
        
        
        theta_bord2 = LinRange(theta_1+0.0000001, theta_2 , 100)
        L = size(theta_bord2)[1]
        x1_bord2 = zeros(L)
        for i = 1 : L 
            x1_bord2[i] = x_fun(theta_bord2[i])
        end 
        
        theta_discr = cat(theta_bord1 ,theta_bord2 , dims = 1 )
        x1 = cat(x1_bord1 ,x1_bord2 , dims = 1 )*t_deforme /Taille
        
        x2 = x_2*ones(size(theta_discr)[1])*t_deforme /Taille
        
        p = plot(ylabel="theta", xlabel="x", legend=:topright)
        #plot!(p,  x1 , theta_discr , label = "x1")
        #plot!(p,  x2 , theta_discr , label = "x2")
        #plot!(p,  x1_bord1 , theta_bord1 , label = "x1_bord1")
        #plot!(p,  x1_bord2 , theta_bord2 , label = "x1_bord2")
        
        #vline!(p, [0 , x_fun(theta_star)], linestyle=:dash, color=:red, linewidth=2)
        #hline!(p, [theta_fun(0) , theta_star], linestyle=:dash, color=:red, linewidth=2)
        display(p)
        
        

        println( mup )
        println("lannnnn")
        filename0 = @sprintf "%s/density_expansion_nu_julia_%.3f_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
        filename0 = @sprintf "%s/density_expansion_nu_theta_edge_2_julia_%.3f_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
        filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
        println("lafffff")

        function box_expansion(n_fun,theta_tab, x1 , x2 , dt, list_t_save,  npts, gb, ratio_m_hbar)
            tmax = maximum(list_t_save)
            x1 = x1 + 0.0000001*theta_tab
            x2 = x2 + 0.0000001*theta_tab

            t = 0.0000001

            mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
            println(@sprintf "t = %.3f saved" t*Taille)
            filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
            println("saved : ", filename)
            npzwrite(filename, mydensity)
            filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)  Float64(t*Taille)
            println("saved : ", filename)
            npzwrite(filename, x1)
            filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
            println("saved : ", filename)
            npzwrite(filename, x2)

           # p = plot(xlabel="x", ylabel="n", legend=:topright)
            #plot!(p, mydensity[1,:],  mydensity[2,:] )
            #display(p)

            println(@sprintf "t = %.3f " t*Taille)
            while t<=tmax

                x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
                t += dt
                println(@sprintf "t = %.3f saved" t*Taille)

                for ts in list_t_save
                    if ts - 0.0001/Taille <= t < (ts + dt) + 0.0001/Taille
                        #println( ts , t , ts+dt )
                        mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                        println(@sprintf "t = %.3f saved" t*Taille)
                        filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                        println("saved : ", filename)
                        npzwrite(filename, mydensity)
                        filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
                        println("saved : ", filename)
                        npzwrite(filename, x1)
                        filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
                        println("saved : ", filename)
                        npzwrite(filename, x2)
                        #p = plot(xlabel="x", ylabel="n", legend=:topright)
                        #plot!(p, mydensity[1,:],  mydensity[2,:] )
                        #display(p)
                    end
                end

            end
            #p = plot(xlabel="x", ylabel="n", legend=:topright)
            #plot!(p, mydensity[1,:],  mydensity[2,:] )
            #display(p)

            return mydensity[1,:],  mydensity[2,:]
        end

        #p = plot(xlabel="theta", ylabel="nu", legend=:topright)
        #plot!(p, theta_discr,  nu_discr )
        #plot!(p, theta_discr, nu_fun(theta_discr))
        #display(p)

        temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille ,51.0/Taille ]
        temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]
        #temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille , 100.0/Taille ,200.0/Taille,300.0/Taille,400.0/Taille , 500.0/Taille  ]
        temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille ]
        temps_sav = [0.1/Taille , 10/Taille , 20/Taille, 30.0/Taille ]
       
        #temps_sav = [1/Taille]
        dt = 1/Taille

        if fit_expension 
            return box_expansion(nu_fun, theta_discr , x1 , x2 , dt ,temps_sav  , npts , gbar, mass/hbar)
        end

    end
end 

new_date = "test_1"
if !isdir(new_date)
    mkdir(new_date)
end

# Afficher la date formatée
println(new_date)
mup = 23.722 * kB
n_p = 26.6
println(n_p)
T = 346

calcule_deformation = 1 == 1 # si on veut calculer les deformations
fit_deformation = 0 == 1 # si on veut fiter la deformation
    
calcule_expension = 1 == 1 # Si on veut fair les clacule GHD d'expension 
fit_expension = 0 == 1 # si on veut faire un fit sur l'expension 

mode = [calcule_deformation, fit_deformation, calcule_expension, fit_expension]
#pre_model_fit_inhomo(new_date , nom,  theta_discr, n_p , mup ,  T , 10 , gbar  , mode , "None" , "None")

In [ ]:
function pre_model_fit_inhomo_bord(new_date , nom,  theta_discr, n_p , mup ,  T ,  x0 , Taille , gbar  , t_deforme )
    
    
    nom = "theta_edge_1.0_ih1"
    #t_deforme = 18
    
        
    if fit_deformation && n_p != "None"
        mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
        println( "mup : " , mup , "    " ,  mup/kB )
    end
    
    
    
    theta_discr , nu_discr , rho_s_discr , rho_discr ,veff_bord, veff_bord_coup ,
    theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr, nu_theta_discr_moy, rho_s_theta_discr_moy, rho_theta_discr_moy = 
    save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T ,x0, Taille , t_deforme, gbar )

    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)
    filename1 = @sprintf "%s/nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)

    function edge_expansion(n_fun, theta_max, npts, gb, ratio_m_hbar)
        theta_tab = LinRange(-theta_max, theta_max, npts)
        x = 0.0000001*theta_tab
        
        
        t = 0
        mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t))
        println("saved : ", filename)
        npzwrite(filename, mydensity)
        filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(t*Taille)
        #println("saved : ", filename)
        npzwrite(filename, x)

        #p = plot(xlabel="x", ylabel="n", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:] , label = " t = 0.0000001" )
        #display(p)
        
        t = "veff_bord"
        x += veff_bord_coup
        #filename = @sprintf "%s/bord_%s_%.3f_%.3f_%.3f_%s.npz" new_date nom Float64(mup/kB) Float64(T) Float64(Taille) t)
        #println("saved : ", filename)
        #npzwrite(filename, x)
        mydensity = eval_edge_density(x , gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %s saved" t)
        filename = filename0 * @sprintf("%s.npz", t)
        #println("saved : ", filename)
        npzwrite(filename, mydensity)
        

        #p = plot(xlabel="v", ylabel="n", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:]  , label = " t = hydro ")
        #display(p)
    end

    # Créer l'interpolation
    nu_fun = LinearInterpolation(theta_discr, nu_discr)

    # Afficher l'interpolation
    #p1 = plot(xlabel="theta", ylabel="nu", legend=:topright)
    #plot!(p1, theta_discr,  nu_discr , label = "nu_discret" )
    #plot!(p1, theta_discr, nu_fun(theta_discr) , label = "nu_fun(theta_discr)")
    #display(p1)

    edge_expansion(nu_fun, theta_max , npts , gbar, mass/hbar)

    ###Coupure de taile $\ell$ + evolution
    t = "veff_bord"
    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T)
    filename = filename0 * @sprintf("%s.npz", t)
    #filename = @sprintf "%s/density_expansion_nu_edge_julia_%.3f_%.3f_%.3f_%.3f.npz" new_date Float64(mup/kB) Float64(T) Float64(Taille) Float64(t*Taille)
    density = npzread(filename)

    

    x1 = density[1,:]#*t_deforme#/Taille
    

    #plot!(p,  x1 , density[2,:] , label = " t = 18 ")
    #display(p)
    
    return x1 , density[2,:] 

end 
    
function pre_model_fit_inhomo_expansion(new_date , nom,  theta_discr, n_p , mup ,  T ,  x0 , Taille , gbar , x_bord , y_bord , t_deform ,  t_expanssion)
   
    theta_discr0 = theta_discr
    
    nom = "theta_edge_1.0_ih2"
    index_last_underscore = findlast(isequal('_'), nom)
    #println(nom[index_last_underscore+1:end])
    
    #println("saveeeeeeeee")
    #save_nu_rho_rho_s(new_date , nom , theta_discr_2, n_p , mup , T , Taille , gbar )
    theta_discr , nu_discr , rho_s_discr , rho_discr ,veff_bord, veff_bord_coup ,
    theta_star, nu_theta_star_discr, rho_s_theta_star_discr, rho_theta_star_discr, nu_theta_discr_moy, rho_s_theta_discr_moy, rho_theta_discr_moy = 
    save_nu_rho_rho_s(new_date , nom , theta_discr, n_p , mup ,  T ,x0, Taille , t_deform, gbar )


    
    x_fun = LinearInterpolation(theta_discr,x_bord) #unite (\mu m /ms ) 
    theta_fun = LinearInterpolation(x_bord, theta_discr)
    x_1 , x_2= (x0 - Taille/2)/t_deform , (x0 + Taille/2)/t_deform
    #x_1 , x_2 = 0.43393268627490844, 1.7461461601247348#0.38599379698434566 , 1.7092275073793108#7.2307096175967445/t_deforme  , 32.01070961759683/t_deforme
    
    theta_1 , theta_2 = theta_fun(x_1), theta_fun(x_2)
    max_abs_theta = maximum([20 , abs(theta_1) , abs(theta_2)])
    
    theta_bord1 = LinRange(-max_abs_theta, theta_1 , 100)
    x1_bord1 = x_1*ones(size(theta_bord1)[1])
    
    
    theta_bord2 = LinRange(theta_1+0.0000001, theta_2 , 100)
    L = size(theta_bord2)[1]
    x1_bord2 = zeros(L)
    for i = 1 : L 
        x1_bord2[i] = x_fun(theta_bord2[i])
    end 
    
    theta_discr = cat(theta_bord1 ,theta_bord2 , dims = 1 )
    x1 = cat(x1_bord1 ,x1_bord2 , dims = 1 )*t_deform /Taille # unite 1 
    
    x2 = x_2*ones(size(theta_discr)[1])*t_deform /Taille
    
    #p = plot(ylabel="theta", xlabel="x", legend=:topright)
    #plot!(p,  x1 , theta_discr , label = "x1")
    #plot!(p,  x2 , theta_discr , label = "x2")
    #plot!(p,  x1_bord1 , theta_bord1 , label = "x1_bord1")
    #plot!(p,  x1_bord2 , theta_bord2 , label = "x1_bord2")
    
    #vline!(p, [0 , x_fun(theta_star)], linestyle=:dash, color=:red, linewidth=2)
    #hline!(p, [theta_fun(0) , theta_star], linestyle=:dash, color=:red, linewidth=2)
    #display(p)

    #p = plot(xlabel="Position (μm)", ylabel="\\theta (μm/ms)", legend=:topright)   
    #plot!(p,  x_bord*t_deform , theta_discr0, label = "bord")
    #plot!(p,  x1*Taille , theta_discr , label = "x1")
    #plot!(p,  x2*Taille , theta_discr , label = "x2")
    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [x0, x0], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :black)
    #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :blue)
    #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :blue)
    #display(p)

    #p = plot(xlabel="Position (μm)", ylabel="\\theta (μm/ms)", legend=:topright)   
    #plot!(p,  x_bord*t_deform , theta_discr0, label = "bord")
    #plot!(p,  x1*Taille , theta_discr , label = "x1")
    #plot!(p,  x2*Taille , theta_discr , label = "x2")
    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [x0, x0], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :black)
    #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :blue)
    #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(theta_discr0), maximum(theta_discr0)],  linestyle = :dot, color = :blue)
    #display(p)
    
    

    #println("mup :", mup )
    #println("lannnnn")
    filename0 = @sprintf "%s/density_expansion_nu_julia_%.3f_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
    filename0 = @sprintf "%s/density_expansion_nu_theta_edge_2_julia_%.3f_%.3f_%.3f_%.3f_" new_date Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
    filename0 = @sprintf "%s/density_expansion_nu_%s_%.3f_%.3f_%.3f_%.3f_" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)
    #println("lafffff")

    function box_expansion(n_fun,theta_tab, x1 , x2 , dt, list_t_save,  npts, gb, ratio_m_hbar)
        tmax = maximum(list_t_save)
        x1 = x1 + 0.0000000000001*theta_tab
        x2 = x2 + 0.0000000000001*theta_tab

        t = 0.0000000000001

        mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
        println(@sprintf "t = %.3f saved" t*Taille)
        filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
        println("saved : ", filename)
        npzwrite(filename, mydensity)
        filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille)  Float64(t*Taille)
        #println("saved : ", filename)
        npzwrite(filename, x1)
        filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
        #println("saved : ", filename)
        npzwrite(filename, x2)

        #p = plot(xlabel="Position (μm)", ylabel="Density ", legend=:topright)
        #plot!(p, x_bord*t_deform , y_bord , label = "bord")
        #plot!(p, [x0, x0], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :black)
        #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
        #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
        #plot!(p, mydensity[1,:]*Taille,  mydensity[2,:] , label = "bord exp")
        #display(p)

        #p = plot(xlabel="x (1) ", ylabel="Density ", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:] )
        #display(p)

        println(@sprintf "t = %.3f " t*Taille)
        while t<=tmax

            x1, x2 = evol_box_expansion(dt, x1, x2, gb, theta_tab, n_fun)
            t += dt
            println(@sprintf "t = %.3f " t*Taille)

            for ts in list_t_save
                if ts - 0.0001/Taille <= t < (ts + dt) + 0.0001/Taille
                    #println( ts , t , ts+dt )
                    mydensity = eval_density(x1, x2, gb, theta_tab, n_fun, ratio_m_hbar)
                    println(@sprintf "t = %.3f saved" t*Taille)
                    filename = filename0 * @sprintf("%.3f.npz", Float64(t*Taille))
                    println("saved : ", filename)
                    npzwrite(filename, mydensity)
                    filename = @sprintf "%s/bord1_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
                    #println("saved : ", filename)
                    npzwrite(filename, x1)
                    filename = @sprintf "%s/bord2_%s_%.3f_%.3f_%.3f_%.3f_%.3f.npz" new_date nom Float64(mup/kB) Float64(T) Float64(x0) Float64(Taille) Float64(t*Taille)
                    #println("saved : ", filename)
                    npzwrite(filename, x2)
                    #p = plot(xlabel="x", ylabel="n", legend=:topright)
                    #plot!(p, mydensity[1,:],  mydensity[2,:] )
                    #display(p)
                end
            end

        end
        #p = plot(xlabel="x", ylabel="n", legend=:topright)
        #plot!(p, mydensity[1,:],  mydensity[2,:] )
        #display(p)

        return mydensity[1,:],  mydensity[2,:]
    end

    nu_fun = LinearInterpolation(theta_discr0, nu_discr)

    #p = plot(xlabel="theta", ylabel="nu", legend=:topright)
    #plot!(p, theta_discr0,  nu_discr )
    #plot!(p, theta_discr0, nu_fun(theta_discr0))
    #display(p)

    temps_sav = [0.1/Taille, 0.5/Taille, 1/Taille, 1.5/Taille, 2/Taille , 2.5/Taille , 3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille ,51.0/Taille ]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille  ]
    #temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille, 40.0/Taille , 50.0/Taille , 100.0/Taille ,200.0/Taille,300.0/Taille,400.0/Taille , 500.0/Taille  ]
    temps_sav = [ 1/Taille, 2/Taille ,  3/Taille, 4/Taille, 5/Taille, 6/Taille, 7/Taille , 8/Taille , 9/Taille,  10.0/Taille, 20.0/Taille, 30.0/Taille ]
    temps_sav = [0.1/Taille , 10/Taille , 20/Taille, 30.0/Taille ]
    temps_sav = [x / Taille for x in 0:10:t_expanssion]
    #temps_sav = [x / Taille for x in 0:10:2]
    temps_sav[1] = 0.1/Taille

    dt = 1/Taille

    return box_expansion(nu_fun, theta_discr , x1 , x2 , dt ,temps_sav  , npts , gbar, mass/hbar)


end


In [ ]:

function model_prel_mu(T,mu)
    #println(@sprintf "T = %.3f ,mu = %.3f " T mu)
    
    theta_discr_integrer = LinRange(-theta_max, theta_max, 600)
    mup = mu * kB
    beta = [-mup / (kB * T) 0 mass /(kB * T)]
    nu_discr = yangyang(gbar, beta, theta_discr)
    nu_fun = LinearInterpolation(theta_discr, nu_discr)
    thet2 , thet1 , npts_integral = theta_discr_integrer[1], theta_discr_integrer[600] , 600
    lam_discr = LinRange(thet2,thet1,npts_integral)
    nu_discr = nu_fun.(lam_discr)    
    densite = mass/hbar* charge_density(gbar, lam_discr, nu_discr, ones(size(lam_discr)[1]) )
    #println(densite)
    return densite
end

function model_mu(T,mu)
    
    OutDensite = 0*ones(size(T)[1])
    for i in 1:size(T)[1]
        #println(@sprintf "i =, T = %.3f ,mu = %.3f " i T[i] mu[i])
        OutDensite[i] = model_prel_mu(T[i],mu[i])
        #println("OutDensite = " ,OutDensite )
    end
    
    if 1 == 0                                                                    
        # Création du graphique
        p = plot(xlabel="mu", ylabel="densite", legend=:topright)
        # Ajout des points avec des marqueurs personnalisés
        plot!(p, mu, OutDensite, seriestype=:scatter, markersize=12, markerstrokewidth=2, markerstrokecolor=:DarkSlateGrey)
        # Affichage du graphique
        display(p)
    end
    
    #println(OutDensite)
    return OutDensite
    
end

if 1 == 0 
    T = LinRange(10, 600 , 3)
    Densite = 56.46504685*ones(size(T)[1])
    p0 = 50*ones(size(T)[1])
    fit = curve_fit(model_mu, T , Densite , p0 )
    
    mu = coef(fit)
    # Création du graphique
    p = plot(xlabel="mu", ylabel="densite", legend=:topright)
    # Ajout des points avec des marqueurs personnalisés
    plot!(p, mu, model_mu(T,mu), seriestype=:scatter, markersize=12, markerstrokewidth=2, markerstrokecolor=:DarkSlateGrey)
    # Affichage du graphique
    display(p)
    
    # Création du graphique
    p = plot(xlabel="T", ylabel="mu", legend=:topright)
    # Ajout des points avec des marqueurs personnalisés
    plot!(p, T, mu, seriestype=:scatter, markersize=12, markerstrokewidth=2, markerstrokecolor=:DarkSlateGrey)
    # Affichage du graphique
    display(p)
end 

function trier_x_et_y(x::Vector{T}, y::Vector{T}) where T
    # Obtenir les indices qui trient x en ordre croissant
    indices_tries = sortperm(x)
    
    # Trier x en ordre croissant
    x_tries = x[indices_tries]
    
    # Réorganiser y en conséquence
    y_tries = y[indices_tries]
    
    return x_tries, y_tries
end

In [ ]:
n_p = 47.23
Taille = 22.176#22.140228271484375#27.866241455078125#23.818206787109375#24.780000000000086  #mu m 
x0 = 18.339047468776954#18.856991739272907

fit_deformation = 0 == 1 
fit_expension = 1 == 1

if fit_deformation
    x_data = DONNEES[2][1]/18 
    y_data = DONNEES[2][2]
    mask = (x_data .> -25 ) .& (x_data .< 10)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension
    x_data = DONNEES[4][1]/Taille 
    y_data = DONNEES[4][2]
    mask = (x_data .> -25000 ) .& (x_data .< 100000)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension 
    x_data = last(DONNEES)[1]./Taille 
    y_data = last(DONNEES)[2]
end

new_date = date_donnees
# Formater la date selon le format spécifié
#new_date = Dates.format(Dates.today(), "yyyy-mm-dd")
# Creer  le dossier
#new_date = "2024-"
if !isdir(new_date)
    mkdir(new_date)
end

calcule_deformation = 1 == 1 # si on veut calculer les deformations
if fit_deformation
    calcule_expension = 0 == 1 # Si on veut fair les clacule GHD d'expension 
#fit_deformation = 0 == 1 # si on veut fiter la deformation
elseif  fit_expension   
    calcule_expension = 1 == 1 # Si on veut fair les clacule GHD d'expension 
#fit_expension = 1 == 1 # si on veut faire un fit sur l'expension 
end

function model_fit_inhomo_un(x, p)
    T = p[1]
    n_p = 56.6
    mu = hbar*om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)/kB
    println("mu = ", mu)
    mu = coef(curve_fit(model_mu, T*ones(1) , n_p*ones(1) , mu*kB/hbar*ones(1) ))[1]  
    #mup = hbar * om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)   #kg um^2.ms^-1
    println("mu = ", mu)
    println("T = ", T) 
    println("n_p , model_mu(T , mu) ", model_mu(T*ones(1) , mu*ones(1))[1])
    mup = mu*kB
    theta_discr = LinRange(-theta_max, theta_max, npts)


    mode = [calcule_deformation, fit_deformation, calcule_expension, fit_expension]
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, x0, Taille, gbar , mode , "None" ,"None")    

    
    function density_fun(x)
        L = size(x)[1]
        y = ones(L)
        fun = LinearInterpolation(x_simul, y_simul)
        for i in 1:L
            if minimum(x_simul) < x[i] < maximum(x_simul) 
                y[i] = fun(x[i])
            elseif x[i]<minimum(x_simul)
                y[i] = fun(minimum(x_simul))
            else 
                y[i] = fun(maximum(x_simul))
            end
        end 
        return y
    end
    #density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="Position (μm/ms)", ylabel="n", legend=:topright)

    densi = density_fun(x)

    plot!(p, x,  y_data, label = "data")
    plot!(p, x_simul, y_simul , label = "simule")
    plot!(p, x, densi , label = "intepol")
    # Tracer une ligne horizontale en pointillé (par exemple, à y = 0)
    plot!(p, [minimum(x), maximum(x)], [0, 0],  linestyle = :dot, color = :red)

    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    plot!(p, [0, 0], [minimum(densi), maximum(densi)],  linestyle = :dot, color = :blue)

    display(p)
    return densi
end 
#Pkg.add("LsqFit")    
using LsqFit


# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
println("size x dat :"  , size(x_data))
p0 = 600*ones(1) 
println( "size(p0) : " , size(p0))
# Bornes des paramètres
lower_bounds = 0.5 .* p0.*ones(size(p0)[1])
upper_bounds = 1.5 .* p0.*ones(size(p0)[1])
# Afficher les bornes
println("lower_bounds = ", lower_bounds , size(lower_bounds))
println("upper_bounds = ", upper_bounds, size(upper_bounds))
# Effectuer l'ajustement avec bornes
println("3")
println("size x_data : " , size(x_data))
println("size y_data : " , size(y_data))    
#fit = curve_fit(model_fit_inhomo_un, x_data, y_data , p0 ) 




In [ ]:
T = 1256.101#coef(fit)[1] 
#T = coef(fit)[1] 
Taille = 22.176#22.140228271484375#27.866241455078125#23.818206787109375#24.780000000000086  #mu m 
x0 = 18.339047468776954#18.856991739272907

n_p = 47.23
mu = hbar*om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)/kB
println("mu = ", mu)
mu = coef(curve_fit(model_mu, T*ones(1) , n_p*ones(1) , mu*kB/hbar*ones(1) ))[1] 

calcule_deformation = 1 == 1 # si on veut calculer les deformations
fit_deformation = 1 == 0 # si on veut fiter la deformation
    
calcule_expension = 1 == 1 # Si on veut fair les clacule GHD d'expension 
fit_expension = 1 == 1 # si on veut faire un fit sur l'expension 

mode = [calcule_deformation, fit_deformation, calcule_expension, fit_expension]



println("mu = ", mu)
println("T = ", T) 
println("n_p , model_mu(T , mu)", model_mu(T*ones(1) , mu*ones(1))[1])
mup = mu*kB
n_p = "None"
theta_discr = LinRange(-theta_max, theta_max, npts)


#x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup,  T, x0, Taille, gbar , mode , "None" , "None")

In [ ]:
#import Pkg; Pkg.add("QuadGK")
using Plots
using Interpolations
using LsqFit
using Roots
using QuadGK
using Dates



n_p =  56.6


fit_deformation = 0 == 1 
fit_expension = 1 == 1

date_donnees = "2024-02-09"
DONNEES = f_DONNEES(date_donnees)

if fit_deformation
    x_data = DONNEES[2][1]/18 
    y_data = DONNEES[2][2]
    mask = (x_data .> -25 ) .& (x_data .< 10)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension
    x_data = DONNEES[4][1] 
    y_data = DONNEES[4][2]
    mask = (x_data .> -25000 ) .& (x_data .< 100000)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension 
    x_data = last(DONNEES)[1] 
    y_data = last(DONNEES)[2]
end

new_date = date_donnees
# Formater la date selon le format spécifié
#new_date = Dates.format(Dates.today(), "yyyy-mm-dd")
# Creer  le dossier
#new_date = "2024-11"
if !isdir(new_date)
    mkdir(new_date)
end



function model_fit_inhomo_deux(x_data_m, p , n_p , Nat , temp_deform , Temps_exp )
    #T = p[1]
    #x0 = 7.585 
    T , x0 = p[1] , p[2]
    #n_p =  56.6
    #n_p =  47.23
    mu = hbar*om_perp * (sqrt(1 + 4 * n_p * a3D) - 1)/kB
    println("mu = ", mu)
    mu = coef(curve_fit(model_mu, T*ones(1) , n_p*ones(1) , mu*kB/hbar*ones(1) ))[1]  
    println("mu = ", mu)
    println("T = ", T) 
    print("x0 = ", x0)
    println("n_p , model_mu(T , mu) ", model_mu(T*ones(1) , mu*ones(1))[1])
    mup = mu*kB
    theta_discr = LinRange(-theta_max, theta_max, npts)

    Taille = 0 
    temp_deform = 18   
    x_bord, y_bord = pre_model_fit_inhomo_bord(new_date , nom,  theta_discr, n_p , mup ,  T , x0 , Taille,  gbar , temp_deform  )
        
    #Nat = 945.6516293099556 #841.24 945.6516293099556
    
    function f(d)
        #window_size = 3 
        #poly_order = 1
        #y = savgol_filter(y_bord , window_size , poly_order) 
        y = y_bord 
        x = x_bord .* temp_deform
        f_interp = LinearInterpolation(x , y , extrapolation_bc = Line())

        flag_plot = false 
        if flag_plot
            plot(x_bord , y_bord , label = "silumation")
            X , Y , label = DONNEES[3]
            plot!(X , Y , label=label)
            vline!([x0 - d / 2 , x0 + d / 2] , color=:blue, linestyle=:dash, linewidth=1.5 , alpha=0.5) 
            vline!(0 , color=:black, linestyle=:dash, linewidth=1.5 , alpha=0.5) 
            hline!(0 , color=:black, linestyle=:dash, linewidth=1.5 , alpha=0.5) 
            grid!(true)
        end 
        integral_value = quadgk(x -> f_interp(x) , x0 -d/2 , x0 +d/2)[1] 
        return (integral_value - Nat*(100-3.2931429180009575)/100)/ Nat
    end

    Taille = find_zero(d -> f(d) , (0.0 , 50.0 ))
    #Taille = 33.18
    println("Taille :" , Taille)
    
    #p = plot(xlabel="Position (μm)", ylabel="\\theta (μm/ms)", legend=:topright)   
    #plot!(p,  x_bord*temp_deform , theta_discr, label = "bord")
    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [x0, x0], [minimum(theta_discr), maximum(theta_discr)],  linestyle = :dot, color = :black)
    #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(theta_discr), maximum(theta_discr)],  linestyle = :dot, color = :blue)
    #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(theta_discr), maximum(theta_discr)],  linestyle = :dot, color = :blue)
    #display(p)

    #p = plot(xlabel="Position (μm)", ylabel="density (μm)^{-1}", legend=:topright)   
    #plot!(p,  x_bord*temp_deform , y_bord, label = "bord")
    #mask = (x_bord .> (x0 - Taille/2)/temp_deform) .& (x_bord .< (x0 + Taille/2)/temp_deform)
    #plot!(p, x_bord[mask] .* temp_deform, y_bord[mask], label = "bord exp")    
    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [x0, x0], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :black)
    #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
    #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
    #display(p)

    #p = plot(xlabel="Position (μm)", ylabel="density (μm)^{-1}", legend=:topright)   
    #plot!(p, x_bord[mask] .* temp_deform, y_bord[mask], label = "bord")
    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [x0, x0], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :black)
    #plot!(p, [x0-Taille/2, x0-Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
    #plot!(p, [x0+Taille/2, x0+Taille/2], [minimum(y_bord), maximum(y_bord)],  linestyle = :dot, color = :blue)
    #display(p)
    
    
    #Temps_exp = 50 
    
    x_simul , y_simul = pre_model_fit_inhomo_expansion(new_date , nom,  theta_discr, n_p , mup ,  T , x0 , Taille,  gbar  , x_bord , y_bord ,  temp_deform , Temps_exp )
    
    function density_fun(x_data_m)
        L = size(x_data_m)[1]
        y = ones(L)
        fun = LinearInterpolation(x_simul.*Taille, y_simul)
        for i in 1:L
            if minimum(x_simul) < x_data_m[i]/Taille < maximum(x_simul) 
                y[i] = fun(x_data_m[i])
            elseif x_data_m[i]/Taille<minimum(x_simul)
                y[i] = fun(minimum(x_simul*Taille))
            else 
                y[i] = fun(maximum(x_simul*Taille))
            end
        end 
        return y
    end
    #density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="Position μm", ylabel="n", legend=:topright)

    densi = density_fun(x_data_m)

    #println("Taille x_data_m : ", size(x_data_m))
    #println("Taille y_data_m : ", size(y_data_m))

    #println("Taille x_simul.*Taille : ", size(x_simul.*Taille))
    #println("Taille y_simul : ", size(y_simul))

    #println("Taille x_data_m : ", size(x_data_m))
    #println("Taille densi : ", size(densi))
    
    plot!(p, x_data,  y_data, label = "data")
    #plot!(p, x_simul.*Taille, y_simul , label = "simule")
    plot!(p, x_data_m, densi , label = "intepol")
    # Tracer une ligne horizontale en pointillé (par exemple, à y = 0)
    #plot!(p, [minimum(x_data), maximum(x_data)], [0, 0],  linestyle = :dot, color = :red)

    # Tracer une ligne verticale en pointillé (par exemple, à x = 0)
    #plot!(p, [0, 0], [minimum(densi), maximum(densi)],  linestyle = :dot, color = :blue)

    display(p)
    #println( densi ) 

    #println("Indices des NaN : ", findall(isnan, densi))
    #println("Le tableau contient des NaN ? ", any(isnan, densi))

    #densi = x_data_m
    
    return densi
end 


# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
mask = (x_data .> -200) .& (x_data .< 50)
x_data_m, y_data_m = x_data[mask], y_data[mask]

plot!(p, x_data_m, y_data_m )
display(p)
                                                                        
println("size x dat :"  , size(x_data_m))
T , x0 = 600 , 18 
p0 = [T , x0 ]
T , x0 = 325.088 , 7.585 
p0 = [T , x0 ]
#p0 = [T ]
#p0 = [T]
println( "size(p0) : " , size(p0))
# Bornes des paramètres
lower_bounds = 0.1 .* p0.*ones(size(p0)[1])
upper_bounds = 3 .* p0.*ones(size(p0)[1])
# Afficher les bornes
println("lower_bounds = ", lower_bounds , size(lower_bounds))
println("upper_bounds = ", upper_bounds, size(upper_bounds))
# Effectuer l'ajustement avec bornes
println("size x_data_m : " , size(x_data_m))
println("size y_data_m : " , size(y_data_m)) 
println("Taille x_data_m : ", size(x_data_m))
println("Taille y_data_m : ", size(y_data_m))

#densi = model_fit_inhomo_deux(x_data_m, p0)


println("x_data_m contient NaN ? ", any(isnan, x_data_m))
println("y_data_m contient NaN ? ", any(isnan, y_data_m))
println("Tailles : ", size(x_data_m), " et ", size(y_data_m))

#densi = model_fit_inhomo_deux(x_data_m, p0)
#println("Test output : ", densi)
#p = plot(xlabel="Position (μm)", ylabel="density (μm)^{-1}", legend=:topright)   
#plot!(p, x_data_m, densi , label = "simul")
#plot!(p, x_data_m, y_data )
# Tracer une ligne verticale en pointillé (par exemple, à x = 0)
#plot!(p, [x0, x0], [minimum(y_data), maximum(y_data)],  linestyle = :dot, color = :black)
#display(p)

lower_bounds .= max.(lower_bounds, eps())  # Empêche des bornes trop petites
upper_bounds .= max.(upper_bounds, eps())  # Empêche des bornes trop petites

n_p , Nat , temp_deform , Temps_exp = 47.27 , 945.6516293099556 , 18 , 50 

try
    println("Tentative d'ajustement...")
    #fit = curve_fit(model_fit_inhomo_deux, x_data_m, y_data_m, p0 , lower_bounds=lower_bounds, upper_bounds=upper_bounds) ,show_trace=true, verbose=true)
    #fit = curve_fit(model_fit_inhomo_deux, x_data_m, y_data_m, p0, lower=lower_bounds, upper=upper_bounds , args=(n_p , Nat , temp_deform , Temps_exp))
    # Affichage des résultats
    println("Paramètres optimisés : ", fit.param)
    println("Covariance estimée : ", fit.cov)

    # Option pour afficher les informations de trace et verbosité
    println("Optimisation terminée")

    println("Ajustement réussi. Paramètres : ", fit.param)
catch e
    println("Erreur détectée : ", e)
    println("x_data_m contient NaN ? : ", any(isnan, x_data_m))
    println("y_data_m contient NaN ? : ", any(isnan, y_data_m))
    println("p0 contient NaN ? : ", any(isnan, p0) ,  p0)
    println("lower_bounds contient NaN ? : ", any(isnan, lower_bounds) ,  lower_bounds)
    println("upper_bounds contient NaN ? : ", any(isnan, upper_bounds) , upper_bounds)
    rethrow(e)
end






In [ ]:
using Optim
using Plots

# Crée un graphique vide
popty = plot(xlabel="Itération", ylabel="Somme des erreurs", legend=:topright)
Sum = Float64[]  # Initialise un tableau vide pour accumuler les sommes

# Fonction objective
function objective_function(p)
    println("Début de l'optimisation avec les paramètres : ", p)
    
    # Appliquer le modèle avec les paramètres actuels
    y_model = model_fit_inhomo_deux(x_data_m, p)
    
    # Vérification si des NaN sont présents
    if any(isnan, y_model)
        println("NaN détecté dans y_model avec les paramètres : ", p)
        return Inf  # Retourner une valeur infinie si NaN est détecté
    end
    
    # Calcul de la somme des erreurs quadratiques
    summ = sum((y_data_m .- y_model).^2)
    println("Somme des erreurs quadratiques : ", summ)
    
    # Ajouter la somme à la liste Sum
    push!(Sum, summ)
    
    # Mise à jour du graphique à chaque itération
    plot!(popty, 1:length(Sum), Sum, linewidth=2, color=:blue)
    display(popty)
    return summ
end

# Paramètres initiaux pour l'optimisation
p0 = [800.0, 18.0]
lower_bounds = [0.1 * p0[1], 0.1 * p0[2]]  # Bornes inférieures
upper_bounds = [3 * p0[1], 3 * p0[2]]      # Bornes supérieures

# Lancement de l'optimisation en utilisant Fminbox avec LBFGS
#result = optimize(objective_function, lower_bounds, upper_bounds, p0, Fminbox(LBFGS()))
#result = optimize(objective_function, p0, LBFGS())



# Affichage du graphique
#display(popty)

# Affichage des résultats de l'optimisation
println("Résultat de l'optimisation : ", result)


In [ ]:
using Optim
using Plots

# Créer un graphique vide avec des labels
popty = plot3d(xlabel="T", ylabel="x0", zlabel="norme2^2", linewidth=2, color=:blue, legend=:topright)

# Initialisation des tableaux pour accumuler les résultats
XT = []
Yx0 = []
Sum = Float64[]  # Initialise un tableau vide pour accumuler les sommes

# Fonction objective
function objective_function(p)
    println("Début de l'optimisation avec les paramètres : ", p)
    
    # Appliquer le modèle avec les paramètres actuels
    y_model = model_fit_inhomo_deux(x_data_m, p)
    
    # Vérification si des NaN sont présents
    if any(isnan, y_model)
        println("NaN détecté dans y_model avec les paramètres : ", p)
        return Inf  # Retourner une valeur infinie si NaN est détecté
    end
    
    # Calcul de la somme des erreurs quadratiques
    summ = sum((y_data_m .- y_model).^2)
    println("Somme des erreurs quadratiques : ", summ)
    
    # Ajouter la somme à la liste Sum
    push!(XT ,p[1])
    push!(Yx0, p[2])
    push!(Sum, summ)
    
    # Mise à jour du graphique à chaque itération
    plot!(popty, XT, Yx0, Sum, linewidth=2, color=:blue)  # Met à jour le graphique avec les nouvelles données
    display(popty)
    
    return summ
end

# Paramètres initiaux pour l'optimisation
p0 = [800.0, 18.0]
lower_bounds = [0.1 * p0[1], 0.1 * p0[2]]  # Bornes inférieures
upper_bounds = [3 * p0[1], 3 * p0[2]]      # Bornes supérieures

# Lancement de l'optimisation en utilisant LBFGS
#result = optimize(objective_function, p0, LBFGS())

# Définir les options d'optimisation
options = Optim.Options(
    x_tol=1e-8,         # Tolérance sur les changements de x
    f_tol=1e-8,         # Tolérance sur les changements de f
    g_tol=1e-8,         # Tolérance sur le gradient
    show_trace=true,    # Afficher les traces de l'optimisation
    iterations=1000     # Limite du nombre d'itérations
)

# Exemple d'optimisation
#result = optimize(
#    objective_function, # Votre fonction objective
#    p0,                 # Point initial
#    LBFGS(),            # Méthode LBFGS
#    options             # Options configurées
#)

println("Résultat de l'optimisation : ", result)


In [ ]:
using Pkg
#Pkg.update("Optim")
#Pkg.update("LineSearches")

using Optim

using LineSearches  # Importer LineSearches pour accéder à FixedStepSize


# Facteurs d'échelle
scale_T = 0.2  # Échelle pour T
scale_x0 = 2.0 # Échelle pour x0 (car 1 tolérance pour T = 0.5 tolérance pour x0)

# Choisir un backend interactif comme plotly
plotly()  # Permet d'avoir un graphique interactif

# Créer un graphique vide avec des labels
popty = plot3d(xlabel="T", ylabel="x0", zlabel="norme2^2", linewidth=2, color=:blue, legend=:topright)

# Initialisation des tableaux pour accumuler les résultats
XT = []
Yx0 = []
Sum = Float64[]  # Initialise un tableau vide pour accumuler les sommes

# Fonction objective
function objective_function(p_scaled , n_p, Nat, temp_deform, Temps_exp)
    # Décodage des variables mises à l'échelle
    T = p_scaled[2] * scale_T
    x0 = p_scaled[1] * scale_x0
    println("Début de l'optimisation avec les paramètres : ", T ," , " ,  x0 )
    

    # Calcul de la fonction modèle
    y_model = model_fit_inhomo_deux(x_data_m, [T, x0] , n_p, Nat, temp_deform, Temps_exp)
    if any(isnan, y_model)
        return Inf
    end

    # Calcul de la somme des erreurs quadratiques
    summ = sum((y_data_m .- y_model).^2)
    println("Somme des erreurs quadratiques : ", summ)
    
    # Ajouter la somme à la liste Sum
    push!(XT ,T)
    push!(Yx0, x0)
    push!(Sum, summ)
    
    # Mise à jour du graphique à chaque itération
    plot!(popty, XT, Yx0, Sum, linewidth=2, color=:blue)  # Met à jour le graphique avec les nouvelles données
    display(popty)
    return summ
end

# Initialisation des paramètres mis à l'échelle
p0 = [ 7 / scale_x0 , 325.088  / scale_T]  # Mise à l'échelle initiale
#p0 = [325.088 / scale_T]  # Mise à l'échelle initiale
lower_bounds = [ 3 , 0.1 * p0[2]]
upper_bounds = [ 8 , 1.5 * p0[2]]
#lower_bounds = 0.5 .* p0.*ones(size(p0)[1])
#upper_bounds = 3 .* p0.*ones(size(p0)[1])

# Options d'optimisation
options = Optim.Options(
    x_tol=1.0,       # Tolérance commune pour les variables mises à l'échelle
    f_tol=1e-8,      # Tolérance pour la fonction objective
    g_tol=1e-8,      # Tolérance pour le gradient
    show_trace=true  # Afficher les informations de progression
    #iterations=1000,    # Nombre d'itérations maximal
)

# Optimisation
#result = optimize(objective_function, lower_bounds, upper_bounds, p0, Fminbox(LBFGS()), options , GradientDescent(linesearch=Optim.FixedStepSize(0.01)))
# Optimisation avec Gradient Descent et pas fixe
#result = optimize(
#    objective_function,
#    p0,
#    GradientDescent(linesearch=Optim.FixedStepSize(2.0)),  # Pas fixe de 2.0
#    options=options
#)
#result = optimize(
#    objective_function,
#    p0,
#    GradientDescent(step_size=2.0),  # Pas fixe de 2.0
#    options=options
#)
#result = optimize(objective_function, lower_bounds, upper_bounds, p0, 
#                  Fminbox(LBFGS(linesearch=LineSearches.HagerZhang())), 
#                  options)
#result = optimize(objective_function, lower_bounds, upper_bounds, p0,
#            BFGS(),
#            options)

n_p , Nat , temp_deform , Temps_exp = 47.27 , 945.6516293099556 , 18 , 50 
f = x -> objective_function(x, n_p, Nat, temp_deform, Temps_exp)
result = optimize(
    f,  # Fonction lambda
    lower_bounds, 
    upper_bounds, 
    p0, 
    Fminbox(LBFGS(linesearch=LineSearches.HagerZhang())),
    #BFGS(), 
    options
)



# Décodage des résultats
p_optimized_scaled = Optim.minimizer(result)
T_optimized = p_optimized_scaled[1] * scale_T
x0_optimized = p_optimized_scaled[2] * scale_x0

println("Résultat de l'optimisation : T = $T_optimized, x0 = $x0_optimized")


In [ ]:
325.183_-3.187

In [ ]:
function detect_nan_variables()
    println("Scanning for NaN values...")
    for (name, value) in pairs(Main)
        try
            if isa(value, AbstractArray) || isa(value, Number)
                if any(isnan, value)
                    println("Variable '$name' contains NaN")
                end
            elseif isa(value, Dict)
                for (k, v) in pairs(value)
                    if isnan(v)
                        println("Variable '$name' contains NaN in key '$k'")
                    end
                end
            end
        catch e
            println("Skipping variable '$name': ", e)
        end
    end
end

detect_nan_variables()

In [ ]:
x_data = DONNEES[4][1] 
y_data = DONNEES[4][2]

p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data , label = "donné")
plot!(p, x_simul * Taille, y_simul, label = "simul")
display(p)

In [ ]:
Taille = 10  # Exemple de Taille
T = 200      # Exemple de valeur de T

# Générer la liste
result = [x / Taille for x in 0:10:T]

println(result)


In [ ]:
#T = coef(fit)[1] 
T = 795.571439555881

n_p = 56.46504685
mu = coef(curve_fit(model_mu, T*ones(1) , n_p*ones(1) , 30*ones(1) ))[1]  
println("mu = ", mu)
println("T = ", T) 
println("n_p , model_mu(T , mu)", model_mu(T*ones(1) , mu*ones(1))[1])
mup = mu*kB
#n_p = "None"
theta_discr = LinRange(-theta_max, theta_max, npts)

calcule_deformation = 1 == 1 
fit_deformation = 0 == 1 
calcule_expension = 1 == 1
fit_expension = 0 == 1 
mode = [calcule_deformation, fit_deformation, calcule_expension, fit_expension]    
x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, x0, Taille, gbar , mode , "None" ,"None")    
 
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data , label = "donné")
plot!(p, x_simul * Taille, y_simul, label = "simul")
display(p)

In [ ]:
Taille = 24.780000000000086  #mu m 

fit_deformation = 1 == 1 
fit_expension = 0 == 1 # on ne peux pas Tailles est un paramentre , doit faire varier les parametre à deux paramatre peux etre

if fit_deformation
    x_data = DONNEES[2][1]/18 
    y_data = DONNEES[2][2]
    mask = (x_data .> -25 ) .& (x_data .< 10)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension
    x_data = DONNEES[4][1]/Taille 
    y_data = DONNEES[4][2]
    mask = (x_data .> -25000 ) .& (x_data .< 100000)
    x_data = x_data[mask]
    y_data = y_data[mask]
elseif fit_expension 
    x_data = last(DONNEES)[1]./Taille 
    y_data = last(DONNEES)[2]
end

new_date = date_donnees
new_date = "test"
# Creer  le dossier
if !isdir(new_date)
    mkdir(new_date)
end

calcule_deformation = 1 == 1 # si on veut calculer les deformations
if fit_deformation
    calcule_expension = 0 == 1 # Si on veut fair les clacule GHD d'expension 
#fit_deformation = 0 == 1 # si on veut fiter la deformation
elseif  fit_expension   
    calcule_expension = 1 == 1 # Si on veut fair les clacule GHD d'expension 
#fit_expension = 1 == 1 # si on veut faire un fit sur l'expension 
end


function model_fit_inhomo_trois(x, p)
    mup , T , Taille = p[1]*kB , p[2] , p[3]  
    n_p = "None"
    Taille = 24.780000000000086  #mu m On fix car complique de le faire varier
    println("mu = ", mup/kB)
    println("T = ", T)  
    println("Taille = ", Taille)
    theta_discr = LinRange(-theta_max, theta_max, npts) 
    mode = [calcule_deformation, fit_deformation, calcule_expension, fit_expension]    
    x_simul, y_simul = pre_model_fit_inhomo(new_date, nom, theta_discr, n_p, mup, T, x0, Taille, gbar , mode , "None" ,"None")    
 
    #x_simul,y_simul = Float64.(x) , Float64.(y_data)

    x_simul, y_simul = trier_x_et_y(x_simul[:,1], y_simul[:,1])
    
    function density_fun(x)
        L = size(x)[1]
        y = ones(L)
        fun = LinearInterpolation(x_simul, y_simul)
        for i in 1:L
            if minimum(x_simul) < x[i] < maximum(x_simul) 
                y[i] = fun(x[i])
            else 
                y[i] = 0
            end
        end 
        return y
    end
    #density_fun = LinearInterpolation(x_simul, y_simul) 
    p = plot(xlabel="x/Taille", ylabel="n", legend=:topright)

    densi = density_fun(x)

    plot!(p, x,  y_data, label = "data")
    plot!(p, x_simul, y_simul , label = "simule")
    plot!(p, x, densi , label = "intepol")
    display(p)
    return densi
end 
    
using LsqFit


x_data, y_data = trier_x_et_y(x_data[:,1], y_data[:,1])



# Tracer les données
scatter(x_data, y_data, label="Données", xlabel="x", ylabel="y", legend=:topleft)                                                                        
p = plot(xlabel="x", ylabel="n", legend=:topright)
plot!(p, x_data, y_data )
display(p)
                                                                        
println("size x dat :"  , size(x_data))
p0 = [55 , 910.0 , 59] 
println( "size(p0) : " , size(p0))
# Bornes des paramètres
lower_bounds = 0.5 .* p0
upper_bounds = 1.5 .* p0
lower_bounds[3] = p0[1] - 0.1
upper_bounds[3] = p0[1] + 0.1


# Afficher les bornes
println("lower_bounds = ", lower_bounds , size(lower_bounds))
println("upper_bounds = ", upper_bounds, size(upper_bounds))
# Effectuer l'ajustement avec bornes
println("3")
println("size x_data : " , size(x_data))
println("size y_data : " , size(y_data))    
fit = curve_fit(model_fit_inhomo_trois, x_data, y_data , p0 ,  lower_bounds , upper_bounds)    